# Compile the post mt systems from 
Kluska et al. 2021
https://ui.adsabs.harvard.edu/abs/2022A%26A...658A..36K/abstract

###  A population of transition disks around evolved stars: Fingerprints of planets. Catalog of disks surrounding Galactic post-AGB binaries

_Kluska, J. search by orcid ; Van Winckel, H. ; Coppée, Q. search by orcid ; Oomen, G.-M. search by orcid ; Dsilva, K. search by orcid ; Kamath, D. search by orcid ; Bujarrabal, V. search by orcid ; Min, M. search by orcid_


In [1]:
import numpy as np
import pandas as pd
import h5py
import json
import subprocess, re, ast

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR


In [3]:
INPUT_FILE = DATA_DIR / "from_others" / "Kluska_2021_table1.txt"
OUTPUT_FILE = RAW_JSON_DIR / "Kluska2021_postAGB.raw.json"
REFERENCE_BIBCODE = "2022A&A...658A..36K"


def hms_to_deg(hms):
    parts = hms.split(":")
    if len(parts) != 3:
        return None
    try:
        h, m, s = map(float, parts)
        return 15.0 * (h + m / 60.0 + s / 3600.0)
    except ValueError:
        return None


def dms_to_deg(dms):
    parts = dms.split(":")
    if len(parts) != 3:
        return None
    try:
        sign = -1.0 if parts[0].startswith("-") else 1.0
        d = abs(float(parts[0]))
        m = float(parts[1])
        s = float(parts[2])
        return sign * (d + m / 60.0 + s / 3600.0)
    except ValueError:
        return None


def tokenize_with_blanks(raw_line):
    # Preserve blank second-name column by splitting on 2+ spaces.
    cols = [c.strip() for c in re.split(r"\s{2,}", raw_line.rstrip("\n"))]
    # Remove empty trailing chunks but keep internal blanks.
    while cols and cols[-1] == "":
        cols.pop()
    return cols


def all_float_tokens(text):
    return [float(x) for x in re.findall(r"[-+]?\d+(?:\.\d+)?", text)]


def parse_period_ecc_from_tail(tail_tokens):
    # Find "yes/no" marker and parse numerics preceding it.
    yn_idx = None
    for i, tok in enumerate(tail_tokens):
        if tok.lower() in {"yes", "no"}:
            yn_idx = i
            break

    parse_region = tail_tokens[:yn_idx] if yn_idx is not None else tail_tokens
    nums = []
    for tok in parse_region:
        nums.extend(all_float_tokens(tok))

    if not nums:
        return None, None
    if len(nums) == 1:
        return nums[-1], None

    # Typical table format has period then eccentricity as the final two numbers before yes/no.
    period_candidate = nums[-2]
    ecc_candidate = nums[-1]

    if 0.0 <= ecc_candidate <= 1.0 and period_candidate > 1.0:
        return period_candidate, ecc_candidate

    # If the final value is clearly not an eccentricity, keep only period.
    return nums[-1], None


def parse_line_to_schema(raw_line):
    cols = tokenize_with_blanks(raw_line)
    if len(cols) < 5:
        return None

    # First non-empty column may be IRAS id (or blank for continuation rows).
    iras_id = cols[0] if cols[0] else None
    common_name = cols[1] if len(cols) > 1 and cols[1] else None

    # Coordinates are usually in cols[2], cols[3] with spaces; normalize to colon format.
    ra_raw = cols[2] if len(cols) > 2 else ""
    dec_raw = cols[3] if len(cols) > 3 else ""
    ra_hms = re.sub(r"\s+", ":", ra_raw.strip()) if ra_raw else ""
    dec_dms = re.sub(r"\s+", ":", dec_raw.strip()) if dec_raw else ""

    ra_deg = hms_to_deg(ra_hms)
    dec_deg = dms_to_deg(dec_dms)

    # Category appears near col[4] (e.g. Cat. 1 / Uncategorized)
    category_tag = cols[4] if len(cols) > 4 else ""

    # Remaining tokens hold stellar params + period/ecc + flags.
    tail_tokens = cols[5:] if len(cols) > 5 else []
    period_val, ecc_val = parse_period_ecc_from_tail(tail_tokens)

    names = [n for n in [iras_id, common_name] if n]
    if not names:
        return None

    system_name = names if len(names) > 1 else names[0]

    entry = {
        "System Name": system_name,
        "RA": [None, ra_deg, None],
        "Dec": [None, dec_deg, None],
        "Period": [None, period_val, None],
        "Eccentricity": [None, ecc_val, None],
        "M1": [None, None, None],
        "M2": [None, None, None],
        "Mass Function": [None, None, None],
        "M1_sin3i": [None, None, None],
        "M2_sin3i": [None, None, None],
        "evol_type_1": "AGB",
        "evol_type_2": None,
        "obs_type_1": "Post-AGB",
        "obs_type_2": None,
        "system_class": "Post-AGB binary",
        "Detection Method": ["RV"],
        "Reference": [REFERENCE_BIBCODE],
        "Notes": f"Source: Kluska+2021 table 1 ({category_tag}). Raw row: {raw_line.strip()}",
        "Simbad": None,
    }
    return entry


entries = []
with open(INPUT_FILE, "r", encoding="utf-8") as fh:
    for raw in fh:
        raw = raw.rstrip("\n")
        if not raw.strip():
            continue
        parsed = parse_line_to_schema(raw)
        if parsed is not None:
            entries.append(parsed)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FILE, "w", encoding="utf-8") as fh:
    for system in entries:
        fh.write(json.dumps(system, separators=(",", ": "), ensure_ascii=False) + "\n")

print(f"Saved {len(entries)} systems to {OUTPUT_FILE}")
print(json.dumps(entries[0], indent=2) if entries else "No entries parsed")


Saved 85 systems to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/Kluska2021_postAGB.raw.json
{
  "System Name": [
    "IRAS01427+4633",
    "BD+46.442"
  ],
  "RA": [
    null,
    26.445958333333333,
    null
  ],
  "Dec": [
    null,
    46.81693611111111,
    null
  ],
  "Period": [
    null,
    140.82,
    null
  ],
  "Eccentricity": [
    null,
    0.0,
    null
  ],
  "M1": [
    null,
    null,
    null
  ],
  "M2": [
    null,
    null,
    null
  ],
  "Mass Function": [
    null,
    null,
    null
  ],
  "M1_sin3i": [
    null,
    null,
    null
  ],
  "M2_sin3i": [
    null,
    null,
    null
  ],
  "evol_type_1": "AGB",
  "evol_type_2": null,
  "obs_type_1": "Post-AGB",
  "obs_type_2": null,
  "system_class": "Post-AGB binary",
  "Detection Method": [
    "RV"
  ],
  "Reference": [
    "2022A&A...658A..36K"
  ],
  "Notes": "Source: Kluska+2021 table 1 (Cat. 1). Raw row: IRAS01427+4633  BD+46.442           01 45 47.03    +46 49 00.97   